# The Hertzsprung–Russell Diagram with Hipparcos

### From a raw ESA star catalogue to a map of stellar evolution

---

The **Hertzsprung–Russell (HR) diagram** is one of the most important plots in stellar
astrophysics. It plots stars by colour (temperature) against true brightness, and when you
do that, stars fall into clear families that reveal how they are born, live, and die.

**Hipparcos** was ESA's first astrometry mission (1989–1993). It measured precise positions,
distances (parallaxes), and brightnesses for about **118,000 stars** — the catalogue that
defined the modern distance scale before Gaia arrived. This notebook takes that catalogue and
builds an HR diagram from it, step by step, mirroring the Gaia notebook exactly so you can
compare the two missions directly.

**What this notebook covers (every plot is interactive — zoom, pan, hover, drag sliders):**

1. The raw HR diagram, straight from the data
2. A cleaned diagram using only high-precision distances
3. An HR diagram coloured by real stellar temperature, with the main sequence, giants, and white dwarfs labelled
4. A slider playground to change the quality cuts and watch the diagram respond
5. Famous stars placed on the diagram, from the Sun to Betelgeuse
6. An animation of how a Sun-like star evolves and dies across the diagram
7. Theoretical isochrones (lines of constant age) for reading a cluster's age
8. A guided project: rebuild everything with data from a different telescope

**How to run:** click the first code cell and press **Shift+Enter** to run it and move to the
next. Repeat down the notebook. That's it.


## 0. Setup — install and import the tools

Run this cell once. On Google Colab everything except `plotly` and `astroquery` is already
installed; the line below quietly adds anything missing. Wait for the confirmation message
before continuing.

In [ ]:
# If you are on Google Colab or a fresh machine, this installs what's needed.
# It is safe to re-run — it does nothing if the packages already exist.
import sys, subprocess

def _ensure(pkgs):
    import importlib
    missing = []
    for pip_name, import_name in pkgs:
        try:
            importlib.import_module(import_name)
        except ImportError:
            missing.append(pip_name)
    if missing:
        print('Installing:', ', '.join(missing), '...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing], check=False)

_ensure([
    ('pandas','pandas'), ('numpy','numpy'),
    ('plotly','plotly'), ('ipywidgets','ipywidgets'),
    ('matplotlib','matplotlib'), ('astroquery','astroquery'),
])

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, HTML

pd.set_option('display.max_columns', 500)
# Make Plotly render in both Jupyter and Colab
try:
    import plotly.io as pio
    pio.renderers.default = 'colab' if 'google.colab' in sys.modules else 'notebook'
except Exception:
    pass

print('All tools ready. Versions:')
print('   pandas ', pd.__version__)
print('   numpy  ', np.__version__)
import plotly; print('   plotly ', plotly.__version__)


## Warm-up — the Harvard colour system

Before we touch any real telescope data, here is the single idea the whole HR diagram rests
on: **a star's colour tells us its temperature.**

Astronomers sort stars into seven colour classes — the **Harvard spectral sequence**:

> **O B A F G K M**  —  hottest and bluest → coolest and reddest
>
> *(a common mnemonic: "Oh Be A Fine Girl/Guy, Kiss Me")*

The next three cells are quick demos that make up their own numbers, so they run instantly
and need no data download.

### Demo 1 — what each spectral class actually looks like

This draws the seven classes as coloured swatches, from hot blue stars to cool red ones,
using an approximate blackbody colour for each surface temperature.

In [ ]:
import numpy as np
import plotly.graph_objects as go

# The seven Harvard classes with a representative surface temperature (Kelvin)
classes = ['O','B','A','F','G','K','M']
temps   = [40000, 20000, 8500, 6500, 5500, 4000, 3000]
examples= ['Mintaka','Rigel','Vega','Procyon','the Sun','Arcturus','Betelgeuse']

def blackbody_rgb(T):
    """Very rough Kelvin -> RGB just for illustration (not physically exact)."""
    T = T/100.0
    if T <= 66:
        r = 255
        g = 99.47*np.log(T) - 161.12 if T > 0 else 0
    else:
        r = 329.7*((T-60)**-0.1332)
        g = 288.12*((T-60)**-0.0755)
    if T >= 66:
        b = 255
    elif T <= 19:
        b = 0
    else:
        b = 138.52*np.log(T-10) - 305.04
    return tuple(int(np.clip(x,0,255)) for x in (r,g,b))

colors = ['rgb{}'.format(blackbody_rgb(T)) for T in temps]

fig = go.Figure()
for i,(cl,T,ex,col) in enumerate(zip(classes,temps,examples,colors)):
    fig.add_shape(type='rect', x0=i, x1=i+1, y0=0, y1=1,
                  fillcolor=col, line=dict(color='white', width=2))
    fig.add_annotation(x=i+0.5, y=0.5, text=f'<b>{cl}</b>', showarrow=False,
                       font=dict(size=30, color='white' if T<7000 else 'black'))
    fig.add_annotation(x=i+0.5, y=1.12, text=f'{T:,} K', showarrow=False, font=dict(size=12))
    fig.add_annotation(x=i+0.5, y=-0.13, text=ex, showarrow=False, font=dict(size=11, color='#555'))
fig.update_xaxes(visible=False, range=[-0.2, 7.2])
fig.update_yaxes(visible=False, range=[-0.3, 1.3])
fig.update_layout(title='The Harvard spectral sequence: hot & blue (left) -> cool & red (right)',
                  template='plotly_white', height=320, width=900, margin=dict(t=60,b=20))
fig.show()


### Demo 2 — how colour turns into a number

Telescopes don't record "bluish" — they record a **colour index**: brightness in a blue
filter minus brightness in a red filter. Hipparcos uses the classic **B - V** index (Gaia
uses BP - RP, which plays the same role). A small (or negative) index means a hot blue star;
a large index means a cool red star. This demo shows the smooth relationship between colour
index and temperature.

In [ ]:
import plotly.express as px

# A schematic colour-index -> temperature curve (illustrative)
bv = np.linspace(-0.4, 2.0, 300)
# Ballesteros' formula: a clean closed-form B-V -> Teff relation
teff = 4600.0*(1.0/(0.92*bv + 1.7) + 1.0/(0.92*bv + 0.62))

fig = px.line(x=bv, y=teff,
              labels={'x':'Colour index  B - V', 'y':'Surface temperature (K)'},
              title='Colour index -> temperature (the conversion behind the HR diagram)')
fig.update_traces(line=dict(width=4, color='#3b4c9a'))
# mark where each Harvard class roughly sits (B-V values)
marks = {'A':0.0,'F':0.4,'G':0.65,'K':1.0,'M':1.5}
for cl,x in marks.items():
    yy = float(4600.0*(1.0/(0.92*x + 1.7) + 1.0/(0.92*x + 0.62)))
    fig.add_scatter(x=[x], y=[yy], mode='markers+text', text=[cl], textposition='top center',
                    marker=dict(size=10, color='#b9770e'), showlegend=False)
fig.update_layout(template='plotly_white', height=460, width=760)
fig.show()
print('Notice: bluer (smaller B-V) = hotter. This is the x-axis of every HR diagram.')


### Demo 3 — a mini HR diagram with made-up stars

Finally, here's the *shape* we're aiming for, built from a handful of invented stars so you
recognise it when the real Hipparcos version appears. Colour on the x-axis, brightness on the
y (flipped so bright is up). Watch the main sequence, a giant, and a white dwarf appear.

In [ ]:
# made-up demo stars: (name, colour index B-V, absolute magnitude, group)
demo_stars = [
    ('hot blue MS', -0.3, -2.0, 'main sequence'),
    ('white MS',     0.0,  1.0, 'main sequence'),
    ('Sun-like MS',  0.65, 4.8, 'main sequence'),
    ('orange MS',    1.0,  7.5, 'main sequence'),
    ('red dwarf MS', 1.5, 12.0, 'main sequence'),
    ('red giant',    1.2, -0.5, 'giant'),
    ('supergiant',   0.1, -6.0, 'giant'),
    ('white dwarf',  0.0, 12.5, 'white dwarf'),
]
import pandas as pd
d = pd.DataFrame(demo_stars, columns=['name','bp_rp','abs_mag','group'])

fig = px.scatter(d, x='bp_rp', y='abs_mag', color='group', text='name',
                 labels={'bp_rp':'Colour index  B - V','abs_mag':'Absolute magnitude'},
                 title='Mini demo HR diagram (made-up stars) - this is the shape to expect')
fig.update_traces(marker=dict(size=14), textposition='middle right')
fig.update_yaxes(autorange='reversed')
fig.update_xaxes(autorange='reversed')
fig.update_layout(template='plotly_white', height=520, width=820)
fig.show()
print('Bright + blue at top-left, faint + red at bottom-right = the main sequence.')


---
**That's the whole idea.** Colour -> temperature (x-axis), and true brightness (y-axis).
Now let's stop inventing numbers and pull **real stars** from ESA's Hipparcos catalogue. 👇

## 1. Get the data — from ESA's Hipparcos catalogue (via `astroquery`)

The Gaia notebook downloaded a CSV chunk from ESA's servers. For Hipparcos the cleanest route
is **`astroquery`**, which pulls the full catalogue (`I/239/hip_main`) straight from the
VizieR database — no website clicking, no manual CSV.

The cell below tries that live download first. If the network is blocked (some networks
restrict it), it falls back to a **realistic synthetic Hipparcos-style sample** so the rest of
the notebook still runs end to end — exactly the same safety net the Gaia notebook used.

In [ ]:
def load_hipparcos():
    """Try the real ESA/VizieR Hipparcos catalogue; fall back to a realistic synthetic sample."""
    try:
        from astroquery.vizier import Vizier
        Vizier.ROW_LIMIT = -1                       # no row cap: get every star
        print('Downloading real Hipparcos catalogue (I/239/hip_main) from VizieR ...')
        cat = Vizier.get_catalogs('I/239/hip_main')
        hip = cat[0].to_pandas()
        # Hipparcos column names -> the names the HR-diagram code expects.
        # Vmag = apparent magnitude, B-V = colour, Plx = parallax (mas), e_Plx = its error
        hip = hip.rename(columns={
            'Vmag':  'phot_g_mean_mag',   # apparent magnitude (matches Gaia's column role)
            'B-V':   'bp_rp',             # B-V plays the same role as Gaia's BP-RP
            'Plx':   'parallax',          # parallax in mas
            'e_Plx': 'parallax_error',    # parallax error in mas
        })
        hip = hip[['phot_g_mean_mag','bp_rp','parallax','parallax_error']].dropna()
        print(f'Downloaded {len(hip):,} real Hipparcos stars.')
        hip.attrs['source'] = 'ESA Hipparcos I/239 (live VizieR download)'
        return hip
    except Exception as e:
        print('Live download unavailable (', type(e).__name__, ').')
        print('   Generating a realistic Hipparcos-style sample so we can keep going...')
        return _synthetic_hipparcos()

def _synthetic_hipparcos(n=40000, seed=42):
    """A physically-motivated fake catalogue: main sequence + giants + white dwarfs.
    Column names match the real Hipparcos table so all later code is identical.
    Hipparcos is shallower and less precise than Gaia, so the noise is a touch larger."""
    rng = np.random.default_rng(seed)
    n_ms, n_giant, n_wd = int(0.80*n), int(0.18*n), n-int(0.80*n)-int(0.18*n)
    # --- main sequence: colour B-V vs absolute mag follow a slanted band ---
    bv_ms = rng.uniform(-0.35, 1.8, n_ms)
    Mabs_ms = 5.2*bv_ms + 1.2 + rng.normal(0, 0.5, n_ms)
    # --- red giant branch: red colour, bright (low Mabs) ---
    bv_g = rng.normal(1.05, 0.22, n_giant)
    Mabs_g = rng.normal(0.6, 0.9, n_giant) - 0.5*(bv_g-1.0)
    # --- white dwarfs (Hipparcos saw only a handful): blue-ish but very faint ---
    bv_wd = rng.uniform(-0.1, 0.6, n_wd)
    Mabs_wd = 11.5 + 2.5*bv_wd + rng.normal(0, 0.4, n_wd)
    bp_rp = np.concatenate([bv_ms, bv_g, bv_wd])
    Mabs  = np.concatenate([Mabs_ms, Mabs_g, Mabs_wd])
    # Hipparcos reached ~roughly a few hundred pc; give each star a distance
    dist_pc = rng.uniform(5, 500, len(bp_rp))
    parallax = 1000.0/dist_pc                          # mas
    g_app = Mabs - 5 - 5*np.log10(parallax/1000.0)
    # Hipparcos parallax errors are ~1 mas typical -> larger fractional error than Gaia
    parallax_error = np.abs(rng.normal(1.0, 0.5, len(bp_rp))) + 0.2
    hip = pd.DataFrame({
        'phot_g_mean_mag': g_app,
        'bp_rp': bp_rp,
        'parallax': parallax,
        'parallax_error': parallax_error,
    })
    hip = hip.sample(frac=1.0, random_state=seed).reset_index(drop=True)
    hip.attrs['source'] = 'Synthetic Hipparcos-style sample (offline fallback)'
    return hip

df = load_hipparcos()
print('Source:', df.attrs.get('source', 'unknown'))
df.head()


### What are we even looking at?

Each row is one star. The columns that matter for us (renamed to match the Gaia pipeline):

| Column | Meaning |
|---|---|
| `phot_g_mean_mag` | how **bright** the star looks from Earth (apparent magnitude; Hipparcos `Vmag`) |
| `bp_rp` | the star's **colour** (Hipparcos `B-V`). Small = blue/hot, large = red/cool |
| `parallax` | the tiny wobble that tells us the **distance** (in milli-arcseconds; `Plx`) |
| `parallax_error` | the uncertainty on that distance (`e_Plx`) |

The trick: *apparent* brightness depends on distance. A faint-looking star might just be far
away. To compare stars fairly we convert to **absolute magnitude** — how bright each star
*truly* is.

In [ ]:
# Absolute magnitude from apparent magnitude + parallax (the distance modulus).
# parallax is in milli-arcsec, so distance(pc) = 1000/parallax.
df['abs_mag'] = df['phot_g_mean_mag'] + 5 + 5*np.log10(df['parallax']/1000.0)
df['distance_pc'] = 1000.0/df['parallax']

print('Added two columns: abs_mag (true brightness) and distance_pc (distance in parsecs).')
df[['bp_rp','phot_g_mean_mag','parallax','abs_mag','distance_pc']].describe().round(2)


## 2. The raw HR diagram

Plot **colour** (x) against **absolute magnitude** (y). One convention: brighter stars have
*smaller* magnitude numbers, so we **flip the y-axis** — bright stars on top, faint on the
bottom, exactly how the sky is arranged.

👉 This plot is interactive: **drag to zoom**, double-click to reset, hover for star details.

In [ ]:
plot_df = df.dropna(subset=['bp_rp','abs_mag'])
plot_df = plot_df[np.isfinite(plot_df['abs_mag'])]
# downsample for snappy interactivity if the catalogue is large
sample = plot_df.sample(min(25000, len(plot_df)), random_state=0)

fig = px.scatter(
    sample, x='bp_rp', y='abs_mag',
    title='Raw Hertzsprung-Russell Diagram (all stars, no quality cuts)',
    labels={'bp_rp':'Colour  B - V   (blue <- -> red)',
            'abs_mag':'Absolute magnitude (bright up)'},
    opacity=0.4,
)
fig.update_traces(marker=dict(size=2.5, color='#3b82f6'))
fig.update_yaxes(autorange='reversed')
fig.update_layout(template='plotly_white', height=650, width=850)
fig.show()


Notice the smear? Lots of stars scatter all over the place. That's not real astrophysics —
it's **noise** from stars whose distances we don't actually know well. Hipparcos is less
precise than Gaia, so this smear is even more pronounced. Time to clean up.

## 3. Clean it up — keep only well-measured distances

A distance is only trustworthy if the parallax error is a small fraction of the parallax.
For Gaia the standard cut is 5%; **Hipparcos is less precise, so 10% is the sensible cut**
(`parallax_error / parallax < 0.10`, and positive parallax). Watch the famous shape of the
**main sequence** snap into focus.

In [ ]:
good = (df['parallax'] > 0) & (df['parallax_error']/df['parallax'] < 0.10)
clean = df[good].dropna(subset=['bp_rp','abs_mag'])
clean = clean[np.isfinite(clean['abs_mag'])]
print(f'Kept {len(clean):,} of {len(df):,} stars ({100*len(clean)/len(df):.1f}%) after the 10% cut.')

s2 = clean.sample(min(25000, len(clean)), random_state=0)
fig = px.scatter(
    s2, x='bp_rp', y='abs_mag',
    title='Cleaned HR Diagram - parallax error < 10%',
    labels={'bp_rp':'Colour  B - V', 'abs_mag':'Absolute magnitude'},
    opacity=0.5,
)
fig.update_traces(marker=dict(size=2.5, color='#6366f1'))
fig.update_yaxes(autorange='reversed')
fig.update_layout(template='plotly_white', height=650, width=850)
fig.show()


## 4. The real HR diagram — coloured by temperature

Now the showpiece. We:
- convert Hipparcos `B - V` colour into an approximate **surface temperature** (in Kelvin),
- colour every star by that temperature using a true blackbody-like palette (blue=hot,
  red=cool),
- and annotate the three great families of stars.

This is the diagram that won Hertzsprung and Russell their place in history — built by *you*
from raw spacecraft data.

In [ ]:
# Ballesteros' formula: a clean closed-form B-V -> effective temperature relation.
# (Good enough to teach with; professionals use fuller calibrations.)
def bv_to_teff(bv):
    bv = np.clip(bv, -0.4, 2.0)
    return 4600.0*(1.0/(0.92*bv + 1.7) + 1.0/(0.92*bv + 0.62))

clean = clean.copy()
clean['teff'] = bv_to_teff(clean['bp_rp'])
s3 = clean.sample(min(30000, len(clean)), random_state=1)

fig = px.scatter(
    s3, x='bp_rp', y='abs_mag', color='teff',
    color_continuous_scale='RdYlBu',  # red(cool) -> blue(hot)
    range_color=(3000, 10000),
    labels={'bp_rp':'Colour  B - V', 'abs_mag':'Absolute magnitude',
            'teff':'T_eff (K)'},
    title='The Hertzsprung-Russell Diagram, coloured by stellar surface temperature',
    hover_data={'distance_pc':':.0f','teff':':.0f','bp_rp':':.2f','abs_mag':':.2f'},
)
fig.update_traces(marker=dict(size=3))
fig.update_yaxes(autorange='reversed')
fig.update_xaxes(autorange='reversed')  # hot/blue on the LEFT, classic orientation

# annotate the three great stellar families
fig.add_annotation(x=0.0, y=2.0, text='Main sequence', showarrow=True, arrowhead=2,
                   ax=60, ay=-40, font=dict(size=13, color='#222'))
fig.add_annotation(x=1.1, y=0.0, text='Giants', showarrow=True, arrowhead=2,
                   ax=40, ay=-30, font=dict(size=13, color='#222'))
fig.add_annotation(x=0.1, y=12.0, text='White dwarfs', showarrow=True, arrowhead=2,
                   ax=60, ay=-30, font=dict(size=13, color='#222'))
fig.update_layout(template='plotly_white', height=680, width=900)
fig.show()


> **Note:** the main sequence isn't a stage of *age* — it's where a star spends about 90%
> of its life fusing hydrogen. A star's position *along* it is set mostly by its **mass**.
> The giants and white dwarfs are the same kinds of stars at later points in their lives.

## 5. The playground — move the sliders

Drag the sliders and the HR diagram redraws instantly. A few things to try:

- tighten the **parallax-quality** cut and watch the noise disappear
- limit the **distance** to keep only nearby stars
- change the **point size and opacity** for a denser or sparser look

In [ ]:
# --- Robust interactive playground (works in Jupyter AND Google Colab) ---
import plotly.graph_objects as go
from ipywidgets import interactive_output, FloatSlider, IntSlider, VBox, Output
from IPython.display import display

base = df.dropna(subset=['bp_rp','abs_mag']).copy()
base = base[(base['parallax'] > 0) & np.isfinite(base['abs_mag'])]
base['teff'] = bv_to_teff(base['bp_rp'])

# In Google Colab, enable the custom widget manager so plots render.
try:
    import google.colab  # noqa
    from google.colab import output as _colab_output
    _colab_output.enable_custom_widget_manager()
except Exception:
    pass

out = Output()

def hr_playground(max_par_err_pct=10.0, max_distance_pc=500, point_size=3, opacity=0.6):
    m = (base['parallax_error']/base['parallax'] < max_par_err_pct/100.0) & \
        (base['distance_pc'] <= max_distance_pc)
    d = base[m]
    if len(d) > 30000:
        d = d.sample(30000, random_state=0)
    fig = go.Figure(go.Scattergl(
        x=d['bp_rp'], y=d['abs_mag'], mode='markers',
        marker=dict(size=point_size, opacity=opacity, color=d['teff'],
                    colorscale='RdYlBu', cmin=3000, cmax=10000,
                    colorbar=dict(title='T (K)')),
        hovertemplate='colour=%{x:.2f}<br>abs mag=%{y:.2f}<extra></extra>',
    ))
    fig.update_yaxes(autorange='reversed', title='Absolute magnitude')
    fig.update_xaxes(autorange='reversed', title='Colour  B - V')
    fig.update_layout(template='plotly_white', height=620, width=850,
                      title=f'{len(d):,} stars  |  err<{max_par_err_pct:.1f}%  |  d<{max_distance_pc} pc')
    with out:
        out.clear_output(wait=True)
        fig.show()

s_err  = FloatSlider(value=10, min=1, max=50, step=1, description='max err %')
s_dist = IntSlider(value=500, min=20, max=500, step=20, description='max dist pc')
s_size = IntSlider(value=3, min=1, max=8, step=1, description='size')
s_op   = FloatSlider(value=0.6, min=0.1, max=1.0, step=0.1, description='opacity')

ui = VBox([s_err, s_dist, s_size, s_op])
display(ui, out)
interactive_output(hr_playground, dict(
    max_par_err_pct=s_err, max_distance_pc=s_dist,
    point_size=s_size, opacity=s_op))

hr_playground()        # draw once up front so a plot is visible


## 6. Where do the famous stars sit?

After the full crowd, here are a few well-known stars — the ones you can pick out in the
night sky. Where each one lands on the diagram tells you immediately what kind of star it is.
(Colours here are given as **B - V** to match the Hipparcos x-axis.)

In [ ]:
# A few well-known stars with their approximate colour (B-V) and absolute
# magnitude, plus a one-line description of each.
famous_stars = [
    # name,            B-V,   abs_mag, short description
    ('The Sun',          0.65,  4.83, 'Our home star - an ordinary main-sequence star'),
    ('Sirius A',         0.00,  1.45, 'Brightest star in our night sky; hot, white, nearby'),
    ('Betelgeuse',       1.85, -5.85, 'Red supergiant in Orion - may explode as a supernova'),
    ('Rigel',           -0.03, -7.0,  'Blue supergiant - thousands of times brighter than the Sun'),
    ('Vega',             0.00,  0.58, 'Bright blue-white star; was the old North Star'),
    ('Proxima Centauri', 1.90, 15.6,  'Closest star to the Sun - a tiny, faint red dwarf'),
    ('Sirius B',        -0.03, 11.3,  'A white dwarf - a dead stellar core the size of Earth'),
]
fam = pd.DataFrame(famous_stars, columns=['name','bp_rp','abs_mag','description'])

# Plot the real Hipparcos cloud faintly in the background, then drop the famous stars on top.
bg = clean.sample(min(20000, len(clean)), random_state=2)

fig = go.Figure()
fig.add_trace(go.Scattergl(
    x=bg['bp_rp'], y=bg['abs_mag'], mode='markers',
    marker=dict(size=2.5, color='lightgray', opacity=0.45),
    name='Hipparcos stars', hoverinfo='skip'))
fig.add_trace(go.Scatter(
    x=fam['bp_rp'], y=fam['abs_mag'], mode='markers+text',
    text=fam['name'], textposition='top center',
    textfont=dict(size=12, color='#111'),
    marker=dict(size=15, color=fam['bp_rp'], colorscale='RdYlBu_r',
                cmin=-0.2, cmax=2, line=dict(width=1.5, color='black')),
    customdata=fam['description'],
    hovertemplate='<b>%{text}</b><br>%{customdata}<extra></extra>',
    name='Famous stars'))
fig.update_yaxes(autorange='reversed', title='Absolute magnitude (bright up)')
fig.update_xaxes(autorange='reversed', title='Colour  B - V  (blue/hot <-  -> red/cool)')
fig.update_layout(template='plotly_white', height=680, width=900,
                  title='Famous stars on the HR diagram (hover for details)',
                  legend=dict(x=0.02, y=0.02))
fig.show()

# Print the descriptions too, so they're visible without hovering.
print('Famous stars in this plot:')
for _, r in fam.iterrows():
    print(f"  {r['name']:<18} B-V={r['bp_rp']:+.2f}  M={r['abs_mag']:+.2f}  - {r['description']}")


**Read the plot like an astronomer:**
- **Sirius A, Vega** sit on the upper main sequence — hot, bright, blue-white.
- **The Sun** sits modestly in the middle of the main sequence.
- **Proxima Centauri** is bottom-right — cool, red, and very faint.
- **Betelgeuse & Rigel** float top-right/top-left of everything — supergiants, off the main sequence.
- **Sirius B** hides at bottom-left — hot but tiny and faint: a white dwarf.

Same two numbers — colour and brightness — and you can already classify any star.

## 7. Animation — the life and death of a Sun-like star

The cell below renders a short MP4 of how a star like the Sun travels across the HR diagram
over billions of years: from a steady main-sequence star, to a swollen red giant, to a
puffed-off planetary nebula, and finally to a slowly cooling white dwarf. The track is drawn
in the **B - V** colour Hipparcos uses.

Run the cell, wait a few seconds while it renders, and the video plays inline.

In [ ]:
# Builds an MP4 of a Sun-like (1 solar mass) star's journey across the HR diagram,
# then plays it inline. Physically ordered: main sequence -> giant -> planetary
# nebula -> cooling white dwarf, with a temperature-coloured fading trail.
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter
from matplotlib.collections import LineCollection
from matplotlib.colors import Normalize
from IPython.display import HTML, display

# --- evolutionary phases for a 1 solar-mass star (colour B-V, absolute mag) ---
phases = [
    ("Main sequence - fusing hydrogen (today)",
        [(0.65, 4.83), (0.68, 4.70), (0.72, 4.45)]),
    ("Subgiant - hydrogen burning in a shell",
        [(0.72, 4.45), (0.85, 3.70), (0.98, 3.00)]),
    ("Red giant branch - swelling and cooling",
        [(0.98, 3.00), (1.15, 1.60), (1.35, 0.20), (1.45, -0.60)]),
    ("Helium flash - core helium ignites",
        [(1.45, -0.60), (1.15, 0.55), (0.95, 0.75)]),
    ("Horizontal branch - steady helium burning",
        [(0.95, 0.75), (1.02, 0.65), (1.12, 0.55)]),
    ("Asymptotic giant - huge and luminous",
        [(1.12, 0.55), (1.30, -0.60), (1.45, -1.60)]),
    ("Planetary nebula - outer layers ejected",
        [(1.45, -1.60), (0.90, -1.00), (0.25, -0.20), (-0.20, 1.50)]),
    ("White dwarf - cooling and fading forever",
        [(-0.20, 1.50), (-0.20, 5.00), (0.00, 8.50), (0.25, 11.50), (0.45, 13.20)]),
]

# Build one dense, smooth polyline through all phases (numpy only, no scipy)
xs, ys, seg, labels = [], [], [], []
per_phase = 28
for pi, (lab, pts) in enumerate(phases):
    labels.append(lab)
    pts = np.array(pts, float)
    t  = np.linspace(0, 1, len(pts))
    tf = np.linspace(0, 1, per_phase)
    xs.append(np.interp(tf, t, pts[:, 0]))
    ys.append(np.interp(tf, t, pts[:, 1]))
    seg.append(np.full(per_phase, pi))
fx = np.concatenate(xs); fy = np.concatenate(ys); seg = np.concatenate(seg)
N = len(fx)

# Star's colour along the track, from B-V via the same Teff relation
teff_track = 4600.0*(1.0/(0.92*np.clip(fx,-0.4,2.0) + 1.7) + 1.0/(0.92*np.clip(fx,-0.4,2.0) + 0.62))

fig, ax = plt.subplots(figsize=(8, 6.4))
ax.set_xlim(-0.6, 1.8); ax.set_ylim(15.5, -8.5)   # mag axis flipped (bright up)
ax.set_xlabel('Colour  B - V  (blue/hot <-  -> red/cool)')
ax.set_ylabel('Absolute magnitude (bright up)')
ax.set_title('Life and death of a Sun-like star (Hipparcos colours)')
ax.grid(alpha=0.2)

norm = Normalize(3000, 10000)
cmap = plt.cm.RdYlBu
head, = ax.plot([], [], 'o', ms=14, mec='k', zorder=5)
phase_text = ax.text(0.03, 0.04, '', transform=ax.transAxes, fontsize=12,
                     bbox=dict(boxstyle='round', fc='white', ec='gray', alpha=0.9))
trail = LineCollection([], cmap=cmap, norm=norm, linewidth=3)
ax.add_collection(trail)

def update(i):
    pts = np.column_stack([fx[:i+1], fy[:i+1]])
    if len(pts) > 1:
        segs = np.concatenate([pts[:-1, None, :], pts[1:, None, :]], axis=1)
        trail.set_segments(segs)
        trail.set_array(teff_track[:i])
    head.set_data([fx[i]], [fy[i]])
    head.set_color(cmap(norm(teff_track[i])))
    phase_text.set_text(labels[seg[i]])
    return head, trail, phase_text

anim = FuncAnimation(fig, update, frames=N, interval=90, blit=False)

try:
    writer = FFMpegWriter(fps=12, bitrate=1800)
    anim.save('star_life.mp4', writer=writer)
    plt.close(fig)
    from base64 import b64encode
    mp4 = open('star_life.mp4','rb').read()
    display(HTML(f'<video width="640" controls autoplay loop>'
                 f'<source src="data:video/mp4;base64,{b64encode(mp4).decode()}" type="video/mp4"></video>'))
except Exception as e:
    print('FFmpeg not available (', type(e).__name__, ') - showing interactive JS animation instead.')
    from IPython.display import HTML as _H
    display(_H(anim.to_jshtml()))
    plt.close(fig)


> A star doesn't drift smoothly across the whole diagram. It spends almost its entire life
> on the main sequence, then moves quickly (in cosmic terms) up into the giant region, sheds
> its outer layers, and ends as a cooling white dwarf in the lower left. The HR diagram is,
> in effect, a map of stellar fate.

## 8. Bonus — read a cluster's age with isochrones

An **isochrone** is the HR-diagram track of a group of stars that were all born at the same
time. As a cluster ages, its most massive (bluest) stars die first, so the top of the main
sequence peels away — the **turn-off** point slides down and to the right. The position of
that turn-off tells you the cluster's age.

The cell below overlays a few schematic isochrones on the data. They are illustrative tracks
for intuition, not a fitted stellar model.

In [ ]:
def schematic_isochrone(age_gyr, n=200):
    """Toy isochrone: a main sequence that turns off earlier (bluer) for younger ages."""
    # turn-off colour gets redder (larger B-V) with age
    turnoff = 0.15 + 0.30*np.log10(age_gyr*1000)  # crude but monotonic
    bv = np.linspace(-0.3, 1.8, n)
    Mabs = 5.2*bv + 1.2  # the main-sequence line from the synthetic model
    # above the turn-off, bend up toward the giant branch
    giant = bv > turnoff
    Mabs[giant] = Mabs[giant] - (bv[giant]-turnoff)*6.0
    return bv, Mabs

s4 = clean.sample(min(15000, len(clean)), random_state=5)
fig = go.Figure()
fig.add_trace(go.Scattergl(x=s4['bp_rp'], y=s4['abs_mag'], mode='markers',
              marker=dict(size=2.5, color='lightgray', opacity=0.5), name='Hipparcos stars'))

colors = px.colors.sequential.Plasma
for i, age in enumerate([0.1, 0.5, 1, 5, 10]):
    bv, M = schematic_isochrone(age)
    fig.add_trace(go.Scatter(x=bv, y=M, mode='lines',
                  line=dict(width=3, color=colors[int(i/4*(len(colors)-1))]),
                  name=f'{age} Gyr'))

fig.update_yaxes(autorange='reversed', title='Absolute magnitude')
fig.update_xaxes(title='Colour  B - V')
fig.update_layout(template='plotly_white', height=680, width=880,
                  title='Schematic isochrones over the Hipparcos HR diagram - older = redder turn-off',
                  legend_title='Age')
fig.show()


## 9. Challenge — extend the notebook

The pipeline works end to end. Here are a few ways to take it further:

1. **The improved catalogue.** Swap `I/239/hip_main` for **`I/311/hip2`**, the 2007 van
   Leeuwen re-reduction with smaller parallax errors. Does the main sequence get tighter?
2. **Density instead of dots.** Replace the scatter in section 4 with
   `px.density_heatmap(...)` to see where stars pile up.
3. **Find the Sun's twins.** Filter `clean` to stars within 0.1 mag and 0.05 colour of the
   Sun (abs_mag ≈ 4.83, B-V ≈ 0.65) and count them.
4. **Hipparcos vs Gaia.** Load the Gaia notebook's catalogue too and overlay both cleaned
   diagrams — Gaia's will be dramatically sharper.
5. **Better temperature scale.** Improve `bv_to_teff` with a published B-V calibration that
   includes a metallicity term and recolour the diagram.

A starter cell follows.

In [ ]:
# Example: find Sun-like stars in the cleaned Hipparcos catalogue.
sun_like = clean[(clean['abs_mag'].between(4.6, 5.0)) &
                 (clean['bp_rp'].between(0.55, 0.75))]
print(f'Found {len(sun_like):,} Sun-like stars in this sample.')
sun_like[['bp_rp','abs_mag','distance_pc','teff']].head(10).round(2)


---
# Project — compare with yet another telescope

You've now rebuilt the entire Gaia pipeline with **Hipparcos** data — same four columns, same
plots, same physics. The natural next project is to bring in a **third** dataset and put all
three side by side.

### The four columns every HR diagram needs

| Role | What it is | Gaia name | Hipparcos name |
|------|-----------|-----------|----------------|
| brightness | apparent magnitude | `phot_g_mean_mag` | `Vmag` |
| colour | colour index (x-axis) | `bp_rp` (BP-RP) | `B-V` |
| distance | parallax in mas | `parallax` | `Plx` |
| quality | parallax error | `parallax_error` | `e_Plx` |

Whatever catalogue you pick, the task is to find these four columns and rename them to match,
so the rest of the code works unchanged — exactly what we did with `Vmag`, `B-V`, `Plx`,
`e_Plx` above.

### Easiest path — the improved Hipparcos-2 catalogue via `astroquery`

The van Leeuwen (2007) re-reduction, `I/311/hip2`, has substantially smaller parallax errors.
The cell below downloads it and sets up the same columns, so you can re-run sections 2–8 on
sharper data and watch the main sequence tighten.

In [ ]:
# Download Hipparcos-2 (van Leeuwen 2007) and set up the same columns as the pipeline.
import numpy as np, pandas as pd

try:
    from astroquery.vizier import Vizier
    Vizier.ROW_LIMIT = -1
    print('Downloading Hipparcos-2 catalogue (I/311/hip2) ...')
    cat = Vizier.get_catalogs('I/311/hip2')
    hip2 = cat[0].to_pandas()
    print(f'Got {len(hip2):,} Hipparcos-2 stars.')
    print('Columns available:', list(hip2.columns))

    # I/311/hip2 carries Plx, e_Plx and a Hipparcos magnitude (Hpmag).
    # It does NOT carry B-V, so for colour we fall back to whatever colour column
    # your download exposes; otherwise reuse the B-V from I/239 by HIP number.
    rename = {}
    if 'Plx'   in hip2.columns: rename['Plx']   = 'parallax'
    if 'e_Plx' in hip2.columns: rename['e_Plx'] = 'parallax_error'
    if 'Hpmag' in hip2.columns: rename['Hpmag'] = 'phot_g_mean_mag'
    if 'B-V'   in hip2.columns: rename['B-V']   = 'bp_rp'
    hip2 = hip2.rename(columns=rename)
    print('\nRenamed to:', [c for c in ["phot_g_mean_mag","bp_rp","parallax","parallax_error"] if c in hip2.columns])
    display(hip2.head())
    print('\nIf B-V is missing, merge it in from the I/239 table on the HIP id, then'
          '\nre-run sections 1-8 using `hip2` in place of `df`.')
except Exception as e:
    print('Could not auto-download (', type(e).__name__, ':', e, ')')
    print('No problem - the synthetic fallback from section 1 already lets the whole notebook run.')


### Now it's your turn

You now have the tools to compare missions directly. Reuse the earlier cells almost verbatim to:

1. apply a quality cut (Hipparcos-2 supports a tighter cut than Hipparcos-1 — try 5%)
2. plot the raw HR diagram (`x='bp_rp'`, `y='abs_mag'`, with the y-axis flipped)
3. make one interactive or temperature-coloured version
4. write a few sentences comparing your Hipparcos diagram to the Gaia one — which mission gives
   the cleaner main sequence, and why?

Same science, three different telescopes.